In [42]:
import networkx as nx 
import pandas as pd 
import numpy as np
import pickle
import os
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
# scipy dependency for PPR

from statsmodels.stats.multitest import multipletests

from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from rpy2.robjects.vectors import ListVector, StrVector

In [43]:
# Load all networks built from previous notebook 

def pickle_load(in_file):
    with open(in_file,'rb') as input_f:
        var = pickle.load(input_f)
    return var

string_nx  = pickle_load("../data/network/string.pkl")
biogrid_nx = pickle_load("../data/network/biogrid.pkl")
bioplex_nx = pickle_load("../data/network/bioplex.pkl")
humanet_nx = pickle_load("../data/network/humannet.pkl")


In [44]:
seed_nodes = pd.read_csv("../cfg/seed_nodes_CP_sym.txt", header=None).values.flatten()
print(seed_nodes)

['PTPRD' 'NTM' 'LAPTM4A' 'SDC1' 'KCNQ3' 'PTK7' 'CUL9']


In [45]:
def ppr_network(seed_nodes, network, name):
    common_nodes = set(seed_nodes) & set(network.nodes)
    print(f"{len(common_nodes)} nodes in common, i.e.: {common_nodes}")

    # build personalized
    personalization = {
        seed: 1/len(common_nodes) for seed in common_nodes
    }

    assert len(set(network.nodes)) == len(list(network.nodes))

    ppr = nx.pagerank(
        G=network,
        personalization=personalization,
    )

    mean_ = np.mean(list(ppr.values()))
    std_ = np.std(list(ppr.values()))

    ppr_df = pd.DataFrame.from_dict(ppr, orient='index', columns=['ppr_importance'])
    ppr_df['zscore'] = (ppr_df['ppr_importance'] - mean_)/std_
    ppr_df = ppr_df.sort_values(['ppr_importance'], ascending=False)
    ppr_df['rank'] = list(range(1,len(ppr_df)+1))
    ppr_df = ppr_df.add_suffix('_'+name)
    return ppr_df
        

In [46]:
networks = {
    'string':   string_nx,
    'biogrid':  biogrid_nx,
    'bioplex':  bioplex_nx,
    'humannet': humanet_nx
}

ppr_results = {}

for k,v in tqdm(networks.items()):
    print(v)
    ppr_results[k] =  ppr_network(seed_nodes, v, k)


  0%|          | 0/4 [00:00<?, ?it/s]

Graph with 11422 nodes and 111137 edges
7 nodes in common, i.e.: {'SDC1', 'CUL9', 'PTK7', 'LAPTM4A', 'KCNQ3', 'NTM', 'PTPRD'}
Graph with 11997 nodes and 99090 edges
6 nodes in common, i.e.: {'SDC1', 'CUL9', 'PTK7', 'LAPTM4A', 'KCNQ3', 'PTPRD'}
Graph with 13957 nodes and 118162 edges
7 nodes in common, i.e.: {'SDC1', 'CUL9', 'PTK7', 'LAPTM4A', 'KCNQ3', 'NTM', 'PTPRD'}
Graph with 18459 nodes and 977495 edges
7 nodes in common, i.e.: {'SDC1', 'CUL9', 'PTK7', 'LAPTM4A', 'KCNQ3', 'NTM', 'PTPRD'}


In [47]:
merged_df = pd.DataFrame()

for k,df in ppr_results.items():
    merged_df = (
        merged_df.merge(
            df,
            how='outer',
            right_index=True,
            left_index=True,
            validate='one_to_one'
        )
    )

In [48]:
# i want genes that are in at least 3 networks
# threshold controls the minimum number of networks

merged_df = merged_df[(merged_df.isna()).sum(axis=1) <= 3] # 3 as there are 3 values for each net and there are 4 networks
merged_df.head()

merged_df.to_parquet("../output/ppr_results.parquet", compression="zstd")

In [49]:
merged_df_ranks = merged_df[['rank_string','rank_biogrid','rank_humannet','rank_bioplex']] #
merged_df_scores = merged_df[['ppr_importance_string','ppr_importance_biogrid','ppr_importance_humannet','ppr_importance_bioplex']] #
merged_df_ranks.head()

,rank_string,rank_biogrid,rank_humannet,rank_bioplex
A1BG,NaN,11366.0,9822.0,8483.0
A1CF,7970.0,9240.0,11871.0,13284.0
A2M,420.0,4650.0,339.0,3613.0
A4GALT,7219.0,NaN,12059.0,7262.0
AAAS,4301.0,6254.0,3911.0,8087.0


In [50]:
r_lists = {}

for col in merged_df_ranks.columns:
    r_lists[col] = StrVector(merged_df_ranks[col].sort_values().dropna().index.tolist())

# Create R list
r_rank_list = ListVector(r_lists)

In [51]:
# Import R package
rankaggreg = importr("RobustRankAggreg")

In [52]:
# Call RankAggreg
result = rankaggreg.aggregateRanks(
    glist = r_rank_list,
    full = True,
    exact = True,
    method = 'RRA'
)
result = pandas2ri.rpy2py(result)

In [53]:
result

,Name,Score
PTPRD,PTPRD,5.480061e-13
LAPTM4A,LAPTM4A,8.129053e-13
PTK7,PTK7,1.017408e-12
SDC1,SDC1,1.733502e-12
KCNQ3,KCNQ3,2.777667e-12
...,...,...
ZNF518B,ZNF518B,9.999747e-01
LMOD3,LMOD3,9.999849e-01
ZNF710,ZNF710,9.999878e-01
ZNF738,ZNF738,9.999905e-01


In [54]:
#top_genes = result.iloc[:int(len(result)*0.01)]
#top_genes.to_csv("../output/top_1perc_genes_RRA.csv")

In [55]:
reject, pvals_fdr, _, _ = multipletests(
    result['Score'],
    alpha=0.05,
    method="fdr_bh"
)


In [56]:
result['FDR'] = pvals_fdr

In [57]:
result[result['FDR'] < 5e-2].to_csv("../output/FDR_significant_genes.csv")